In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import os
import requests
from langchain_community.tools import DuckDuckGoSearchRun

/tmp/ipykernel_13977/1820915120.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [3]:
search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke("What is top news about coin?")

In [4]:
results

'Read the latest crypto market news stories for major cryptocurrency coins, tokens and companies. 24h coverage with new content added throughout the day. 17 hours ago · Latest Bitcoin news today with live updates, summaries, market sentiment, recent price moves, trends, and more from trusted sources. CoinGecko News gives you the best of cryptocurrency and blockchain news in one place. Get updating feed of top news, market insights, and expert opinions across the globe. Guides, explainers, analyses and product comparisons from the world of crypto. PEPE Surges 18% as Meme Coins Rally – Could Maxi Doge Outperform It? Wall Street Power Grab? Anti-Crypto Elizabeth Warren Could Put Day Trading in the Crosshairs. LIVE – Why Is Crypto Down? Jensen Huang Chinese Spy Scandal Sends AI Stocks Into Chaos. Leader in cryptocurrency, Bitcoin, Ethereum, XRP, blockchain, DeFi, digital finance and Web 3.0 news with analysis, video and live price updates.'

In [5]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)


In [6]:
llm.invoke("Hi, how are you?")

AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 13, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 1.995e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.995e-05, 'upstream_inference_prompt_cost': 1.95e-06, 'upstream_inference_completions_cost': 1.8e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_40057a9dcc', 'id': 'gen-1794245660-oXtDbJTvay2tdCCLtjbQ', 'service_tier': 'default', 'finish_reason':

In [7]:
from langsmith import Client
from langchain_classic.agents import AgentExecutor, create_react_agent

client = Client()
prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True,
)

# Legacy alternative:
# from langchain_classic import hub
# prompt = hub.pull("hwchase17/react")

agent = create_react_agent(
    llm=llm,
    tools=[search_tool],
    prompt=prompt,
)
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
)

In [9]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [8]:
response = agent_executor.invoke(
    {"input": "give me three way to reach the moon?"}
)
response["output"]



> Entering new AgentExecutor chain...
To answer this question, I need to consider the various methods or missions that have successfully reached the Moon in the past or are currently being developed. These methods could include manned missions, unmanned missions, and other technological approaches. I'll look for relevant information to provide a well-rounded answer.

Action: duckduckgo_search
Action Input: "ways to reach the moon methods missions"The Soviet Luna programme was the first successful lunar programme, its Luna 1 (1959) being the first partially successful lunar mission The first image taken of the far side of the Moon, returned by Luna 3 (1959) Missions to the Moon have been numerous and represent some of the earliest endeavours in space missions, with continuous exploration of the Moon beginning in 1959. The first ... May 25, 2022 · But it didn’t necessarily have to be this way: There are many ways to get to the Moon, and NASA studied multiple approaches before opting fo

"Three ways to reach the Moon are: 1) Manned missions (e.g., NASA's Apollo and Artemis programs), 2) Unmanned robotic missions (e.g., China's Chang'e missions), and 3) Commercial spacecraft efforts by private companies for lunar exploration."